In [1]:
import pandas as pd
import json
import time
from openai import OpenAI
import os

In [2]:
# This step loads environment variables from a .env file in the parent folder
# the .env contains OPENAI_API_KEY=sk-proj-fhsjkfhsjdfsjdfhhfsd

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# ============================================================
# CONFIG
# ============================================================

API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-5.4"   # or "gpt-5.5"
BATCH_SIZE = 10
MAX_ROWS = None           # set to None for all rows
SLEEP_SECONDS = 1        # helps avoid rate limits

# INPUT_CSV = 'inputs/simple_matches_Intake24_Recall_Items_MF1L_direct.csv'
INPUT_CSV = 'inputs/Y17_MF1L_withAFP_for_Tom_grouped_mismatches.csv'
OUTPUT_CSV = "outputs/Y17_MF1L_withAFP_for_Tom_grouped_mismatches_out.csv"

client = OpenAI(api_key=API_KEY)

In [4]:
# ============================================================
# SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
TASK:
For each case, review the foods mentioned in searched_items and decide whether each one is represented anywhere in recorded_items.

The main question is:
Is each searched food accounted for by something in the recorded foods?

Only after checking coverage for all searched foods should you identify any searched items that are not accounted for.

MATCHING RULES:
- Use broad semantic matching, not exact text matching.
- If the recorded list contains a related, more general, more specific, similar, or approximate food, treat the searched item as accounted for.
- The match can be vague as long as the core food is present.
- Be conservative about declaring an item missing.
- If unsure, prefer "accounted for" over "missing".

EXAMPLES OF ACCOUNTED-FOR MATCHES:
- "special rice" -> "white rice"
- "brie" -> "cheese"
- "curry" -> "chicken madras"
- "coke" -> "pepsi"
- "biscuit" -> "Cadburys Wispa"
- "orange juice" -> "apple squash"

ADDITIONAL GUIDANCE:
- Brand names can match generic foods and similar brands.
- Minor spelling mistakes, abbreviations, and approximate wording should still match when the intended food is clear.
- Differences in flavor, subtype, brand, preparation style, or variety usually do not make an item missing if the core food is represented.
- Compare each searched food individually against the full meaning of recorded_items.
- searched_items and recorded_items are raw text strings and may contain comma-separated items, but do not rely mechanically on commas alone; interpret food phrases semantically.

WHEN TO MARK NOT ACCOUNTED FOR:
- Mark a searched item as missing only if there is no reasonable food in recorded_items that accounts for it.
- Only include foods that are clearly not represented at all.

IMPORTANT:
- BOTH fields are raw text strings and may contain comma-separated items.
- DO NOT mechanically treat commas as definitive item separators.
- Instead, interpret semantically (food phrases may contain commas or qualifiers).
- Compare meaning, not punctuation structure.

Return JSON only.

The response must be a JSON array.
One output object per input case.

Each object must contain:
- case_id
- missing_items
- all_items_accounted_for
- confidence
- reasoning

Keep reasoning brief (1 sentence).
"""

In [5]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def split_foods(text):
    """
    Convert comma-separated string into cleaned list.
    """
    if pd.isna(text):
        return []

    return [
        item.strip()
        for item in str(text).split(",")
        if item.strip()
    ]


def build_batch_payload(batch_df):
    """
    Convert dataframe rows into structured JSON payload.
    """

    payload = []

    for idx, row in batch_df.iterrows():

        payload.append({
            "case_id": int(idx),
            #"searched_items": split_foods(row["SearchTerm"]),
            #"recorded_items": split_foods(row["DescriptionEn"])
            #"searched_items": row["SearchTerm"],
            #"recorded_items": row["DescriptionEn"]
            "searched_items": row["SearchTerm"],
            "recorded_items": row["NameEn"]

        })

    return payload


def analyse_batch(batch_payload):

    user_prompt = json.dumps(batch_payload, indent=2)

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "food_reconciliation_batch",
                "schema": {
                    "type": "object",
                    "properties": {
                        "results": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "case_id": {"type": "integer"},
                                    "missing_items": {
                                        "type": "array",
                                        "items": {"type": "string"}
                                    },
                                    "all_items_accounted_for": {"type": "boolean"},
                                    "confidence": {"type": "number"},
                                    "reasoning": {"type": "string"}
                                },
                                "required": [
                                    "case_id",
                                    "missing_items",
                                    "all_items_accounted_for",
                                    "confidence",
                                    "reasoning"
                                ]
                            }
                        }
                    },
                    "required": ["results"],
                    "additionalProperties": False
                }
            }
        }
    )

    #content = response.choices[0].message.content
    content = json.loads(response.choices[0].message.content)
    results_json = content["results"]

    return results_json

In [9]:
# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(INPUT_CSV)

# ============================================================
# FILTER TO POTENTIAL MISMATCHES
# ============================================================

subset = df[df["potential_mismatch"] == True].copy()

if MAX_ROWS is not None:
    subset = subset.head(MAX_ROWS)

print(f"Rows to process: {len(subset)}")

Rows to process: 19576


In [41]:
batch_df = subset.iloc[0:9]
batch_payload = build_batch_payload(batch_df)


In [54]:
results = analyse_batch(batch_payload)

In [10]:


# ============================================================
# CREATE OUTPUT COLUMNS
# ============================================================

df["llm_missing_items"] = None
df["llm_all_accounted_for"] = None
df["llm_confidence"] = None
df["llm_reasoning"] = None

# ============================================================
# PROCESS IN BATCHES
# ============================================================

num_batches = (len(subset) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_num in range(num_batches):

    start = batch_num * BATCH_SIZE
    end = start + BATCH_SIZE

    batch_df = subset.iloc[start:end]

    print(
        f"Processing batch {batch_num + 1}/{num_batches} "
        f"({len(batch_df)} rows)"
    )

    try:

        batch_payload = build_batch_payload(batch_df)

        results = analyse_batch(batch_payload)

        # --------------------------------------------
        # WRITE RESULTS BACK
        # --------------------------------------------

        for result in results:

            idx = result["case_id"]

            df.at[idx, "llm_missing_items"] = json.dumps(
                result["missing_items"]
            )

            df.at[idx, "llm_all_accounted_for"] = result[
                "all_items_accounted_for"
            ]

            df.at[idx, "llm_confidence"] = result[
                "confidence"
            ]

            df.at[idx, "llm_reasoning"] = result[
                "reasoning"
            ]

        # --------------------------------------------
        # OPTIONAL INTERMEDIATE SAVE
        # --------------------------------------------

        if (batch_num + 1) % 10 == 0:

            temp_name = f"checkpoints/checkpoint_batch_{batch_num+1}.csv"

            df.to_csv(temp_name, index=False)

            print(f"Checkpoint saved: {temp_name}")

        time.sleep(SLEEP_SECONDS)

    except Exception as e:

        print(f"Error in batch {batch_num + 1}")
        print(e)

# ============================================================
# SAVE FINAL OUTPUT
# ============================================================

df.to_csv(OUTPUT_CSV, index=False)

print("Done.")
print(f"Saved to: {OUTPUT_CSV}")

Processing batch 1/1958 (10 rows)
Processing batch 2/1958 (10 rows)
Processing batch 3/1958 (10 rows)
Processing batch 4/1958 (10 rows)
Processing batch 5/1958 (10 rows)
Processing batch 6/1958 (10 rows)
Processing batch 7/1958 (10 rows)
Processing batch 8/1958 (10 rows)
Processing batch 9/1958 (10 rows)
Processing batch 10/1958 (10 rows)
Checkpoint saved: checkpoints/checkpoint_batch_10.csv
Processing batch 11/1958 (10 rows)
Processing batch 12/1958 (10 rows)
Processing batch 13/1958 (10 rows)
Processing batch 14/1958 (10 rows)
Processing batch 15/1958 (10 rows)
Processing batch 16/1958 (10 rows)
Processing batch 17/1958 (10 rows)
Processing batch 18/1958 (10 rows)
Processing batch 19/1958 (10 rows)
Processing batch 20/1958 (10 rows)
Checkpoint saved: checkpoints/checkpoint_batch_20.csv
Processing batch 21/1958 (10 rows)
Processing batch 22/1958 (10 rows)
Processing batch 23/1958 (10 rows)
Processing batch 24/1958 (10 rows)
Processing batch 25/1958 (10 rows)
Processing batch 26/1958 (

In [ ]:
# ============================================================
# SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are an assistant that checks whether searched food items are accounted for in recorded food items.

TASK:
Confirm that all foods mentioned in searched_items
are accounted for in recorded_items with an approximate match.

Core principle:
Use broad, meaning-based matching. Exact matches are NOT required. If the recorded list contains a sufficiently similar or more general/specific food, treat it accounted for.

Missing guidance:
- Match by semantic similarity, not exact wording, to find missing food
- If the recorded list contains a related, more general, more specific, similar, or approximate food, treat the searched item as accounted for
- Examples:
  - "apricot jam" is covered by "fruit conserve"
  - "brie" is covered by "cheese"
  - "chicken curry" is covered by "chicken madras"
  - "coke" is covered by "pepsi"
  - "crisps" is covered by "McCoys crinkle cut"
- Brand names can match generic foods or similar branded products.
- Minor spelling mistakes, abbreviations, and approximate wording should still match when the intended food is clear.
- Preparation style, flavor, brand, and subtype differences usually do NOT make an item missing if the core food is present.
- Only mark an item as missing when the searched food is clearly absent from the recorded foods.

IMPORTANT:
- BOTH fields are raw text strings and may contain comma-separated items.
- DO NOT mechanically treat commas as definitive item separators.
- Instead, interpret semantically (food phrases may contain commas or qualifiers).
- Compare meaning, not punctuation structure.

Return JSON only.

The response must be a JSON array.
One output object per input case.

Each object must contain:
- case_id
- missing_items
- all_items_accounted_for
- confidence
- reasoning

Keep reasoning brief (1 sentence).
"""

In [ ]:
# ============================================================
# SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are an assistant that checks for missing food items in records of food made by study participants searching for the foods they have eaten.

TASK:
Determine whether any foods mentioned in searched_items
are missing from recorded_items.

Core principle:
Use broad, meaning-based matching. Exact matches are NOT required. If the recorded list contains a sufficiently similar or more general/specific food, treat it as covered.

RULES:
- Treat semantically equivalent foods as matches: A search for 'chicken curry' would match 'chicken madras'
- Branded items can match generic foods and similar brands
- A search for coke can match pepsi
- search terms might have spelling mistakes
- approximate matches count like 'cheese' and 'brie'
- Identify ONLY foods clearly missing. Searching black coffee and recording instant coffee is not missing.
- Be conservative about adding missing foods - the match can be quite vauge
- Compare EACH searched item individually

IMPORTANT:
- BOTH fields are raw text strings and may contain comma-separated items.
- DO NOT mechanically treat commas as definitive item separators.
- Instead, interpret semantically (food phrases may contain commas or qualifiers).
- Compare meaning, not punctuation structure.

Return JSON only.

The response must be a JSON array.
One output object per input case.

Each object must contain:
- case_id
- missing_items
- all_items_accounted_for
- confidence
- reasoning

Keep reasoning brief (1 sentence).
"""